# Analysis 1.A — Treasury curve conditions and movements

Inspect 2Y/10Y/30Y Treasury yields and compare exact endpoints. Yield levels are percent; spread levels and all changes are basis points. Start Jupyter from the repository root with `poetry run jupyter lab` after `poetry install`, using the Poetry Python kernel.

Execute **A → B → C → D** manually. After changing either DatePicker, rerun C to redraw the selected period and D to recalculate; each reads the widgets independently. Zoom, pan and the range slider inspect the history without changing selected analysis dates. No automatic callbacks are installed.

`download` retrieves and saves one self-contained CSV with comment-header provenance. The key order is explicit Python argument → `FRED_API_KEY` environment variable → root `.env`; without a key, the three known H.15 series use FRED's public CSV. `file` reads the selected CSV offline. Old JSON caches must be downloaded again or converted explicitly; see [README](README.md).

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

from bondview.data.fred import FredDataManager
from bondview.analysis.treasury_curve import analyze_curve
from bondview.plotting.treasury_curve import show_yield_history
from bondview.presentation.treasury_curve import date_picker_defaults, show_curve_results

# Set the repository root explicitly if starting Jupyter outside the project.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_MODE = "download"  # "download" or "file"
DATA_PATH = PROJECT_ROOT / "data/fred/treasury_yields.csv"
DOWNLOAD_START = "2024-01-01"
DOWNLOAD_END = "2025-01-31"
SLOPE_TOLERANCE_BP = 0.0

## A. Load FRED data

This is the only step that can contact FRED. Missing observations remain missing; the manager retains provenance independently of DataFrame attributes.

In [ ]:
fred = FredDataManager(DATA_PATH, env_path=PROJECT_ROOT / ".env")
observations = fred.load_or_download(DATA_MODE, DOWNLOAD_START, DOWNLOAD_END)
fred.report()

## B. Select dates

Initial values use the loaded date coverage. Dates are exact: weekends, holidays and out-of-coverage dates may produce `N/A`. Changing a widget changes the next execution's inputs; it does not automatically refresh existing outputs. Rerunning B resets the widgets to loaded coverage.

In [ ]:
start_options, end_options = date_picker_defaults(observations)
start_picker = widgets.DatePicker(**start_options)
end_picker = widgets.DatePicker(**end_options)
display(start_picker, end_picker)

## C. Draw Treasury yield history — run manually

Both selected widget dates set this figure's initial visible range. Full loaded history remains available through normal Plotly interactions. Missing observations remain gaps.

In [ ]:
history_figure = show_yield_history(
    observations, start_picker.value, end_picker.value,
)

## D. Calculate and display Analysis 1.A — run manually

Each segment uses `Δslope = Δlong − Δshort`. The nonnegative `SLOPE_TOLERANCE_BP` applies inclusively only to slope changes; its Python default is zero. Yield directions use actual signs. Unequal changes never become parallel shifts; stationary tenors are stated explicitly. Opposing-direction twist is a descriptive convention, not a universal taxonomy.

Configurations use the signs of endpoint spreads. Available endpoints survive missing dependent changes; no nearby observation substitutes for a missing date. The five-row structured result uses stable IDs; presentation terminology comes from `config/labels.yaml`.

The design example interval 2024-09-16 → 2025-01-13 has yield changes +84/+116/+103 bp and spread changes +32/−13 bp. Loaded observations determine these results; agreement is never forced. This is net historical Treasury-tenor analysis, not a forecast, ETF return, or recommendation. Retrieval snapshots do not reconstruct historical release availability.

In [ ]:
result = analyze_curve(
    observations, start_picker.value, end_picker.value,
    slope_tolerance_bp=SLOPE_TOLERANCE_BP,
)
show_curve_results(result)